In [1]:
import pandas as pd

In [2]:
train_dat = pd.read_csv('../Data/ClimateTrain.csv')
train_dat.head()

,date,meantemp,humidity,wind_speed,meanpressure
0,2013-01-01,10.000000,84.500000,0.000000,1015.666667
1,2013-01-02,7.400000,92.000000,2.980000,1017.800000
2,2013-01-03,7.166667,87.000000,4.633333,1018.666667
3,2013-01-04,8.666667,71.333333,1.233333,1017.166667
4,2013-01-05,6.000000,86.833333,3.700000,1016.500000


In [3]:
train_dat.tail()

,date,meantemp,humidity,wind_speed,meanpressure
1457,2016-12-28,17.217391,68.043478,3.547826,1015.565217
1458,2016-12-29,15.238095,87.857143,6.000000,1016.904762
1459,2016-12-30,14.095238,89.666667,6.266667,1017.904762
1460,2016-12-31,15.052632,87.000000,7.325000,1016.100000
1461,2017-01-01,10.000000,100.000000,0.000000,1016.000000


In [4]:
train_dat = train_dat[train_dat['date']!='2017-01-01']

In [5]:
train_dat.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1461 entries, 0 to 1460
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          1461 non-null   object 
 1   meantemp      1461 non-null   float64
 2   humidity      1461 non-null   float64
 3   wind_speed    1461 non-null   float64
 4   meanpressure  1461 non-null   float64
dtypes: float64(4), object(1)
memory usage: 68.5+ KB


In [12]:
train_dat['date'] = pd.to_datetime(train_dat['date'])
train_dat[['meantemp', 'humidity', 'wind_speed', 'meanpressure']] = train_dat.drop(columns=['date'], inplace=False).apply(lambda x: round(x, 4))
train_dat[['meantemp', 'humidity', 'wind_speed', 'meanpressure']] = train_dat.drop(columns=['date'], inplace=False).astype('float32')

In [7]:
train_dat.describe()

,date,meantemp,humidity,wind_speed,meanpressure
count,1461,1461.000000,1461.000000,1461.000000,1461.000000
mean,2015-01-01 00:00:00,25.506127,60.744851,6.806865,1011.101197
min,2013-01-01 00:00:00,6.000000,13.428571,0.000000,-3.041667
25%,2014-01-01 00:00:00,18.857143,50.375000,3.475000,1001.571429
50%,2015-01-01 00:00:00,27.714286,62.625000,6.250000,1008.555556
75%,2016-01-01 00:00:00,31.312500,72.125000,9.250000,1014.937500
max,2016-12-31 00:00:00,38.714286,98.000000,42.220000,7679.333333
std,NaN,7.339416,16.743928,4.559688,180.293335


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --- Configuration ---
TRAIN_FILE = 'ClimateTrain.csv'
TEST_FILE = 'ClimateTest.csv' # <-- New file for testing
LOOK_BACK = 60 # Number of previous days to use for prediction

try:
    # --- 1. Load and Preprocess Data ---
    # Load training data
    df_train = pd.read_csv(TRAIN_FILE)
    df_train['date'] = pd.to_datetime(df_train['date'])
    df_train.set_index('date', inplace=True)
    print(f"Training data loaded successfully. Shape: {df_train.shape}")

    # Load testing data
    df_test = pd.read_csv(TEST_FILE)
    df_test['date'] = pd.to_datetime(df_test['date'])
    df_test.set_index('date', inplace=True)
    print(f"Testing data loaded successfully. Shape: {df_test.shape}")

    # --- 2. Normalize the Data ---
    # IMPORTANT: Fit the scaler ONLY on the training data to prevent data leakage.
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(df_train)

    # Separate scaler for the target variable for inverse transforming
    target_scaler = MinMaxScaler(feature_range=(0, 1))
    target_scaler.fit(df_train[[TARGET_COLUMN]])

    # Apply the transformation to both training and testing data
    scaled_train_data = scaler.transform(df_train)

    # --- 3. Create Training Sequences ---
    X_train, y_train = [], []
    target_col_index = df_train.columns.get_loc(TARGET_COLUMN)

    for i in range(LOOK_BACK, len(scaled_train_data)):
        X_train.append(scaled_train_data[i-LOOK_BACK:i, :])
        y_train.append(scaled_train_data[i, target_col_index])

    X_train, y_train = np.array(X_train), np.array(y_train)
    print(f"Shape of X_train: {X_train.shape}")

    # --- 4. Prepare Test Data Sequences ---
    # To predict the test set, we need the last `LOOK_BACK` days from the training data
    # to form the initial sequences.
    all_features = list(df_train.columns)
    combined_data = pd.concat((df_train[all_features], df_test[all_features]), axis=0)
    
    # Get the input data for the test set
    test_inputs = combined_data[len(df_train) - LOOK_BACK:].values
    
    # Scale the test inputs using the scaler fitted on the training data
    scaled_test_inputs = scaler.transform(test_inputs)

    # Create the test sequences
    X_test = []
    for i in range(LOOK_BACK, len(scaled_test_inputs)):
        X_test.append(scaled_test_inputs[i-LOOK_BACK:i, :])
    
    X_test = np.array(X_test)
    print(f"Shape of X_test: {X_test.shape}")

    # The actual values we want to compare against are in the original test dataframe
    y_test_actual = df_test[TARGET_COLUMN].values

    # --- 5. Build and Train the LSTM Model ---
    # The model architecture remains the same
    model = Sequential([
        LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.2),
        LSTM(units=50, return_sequences=False),
        Dropout(0.2),
        Dense(units=25),
        Dense(units=1)
    ])

    model.compile(optimizer='adam', loss='mean_squared_error')
    model.summary()
    
    # Use a small validation split from the TRAINING data for early stopping
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.1, # Use last 10% of training data for validation
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, verbose=1)],
        verbose=1
    )

    # --- 6. Make Predictions and Evaluate on the Test Set ---
    predictions_scaled = model.predict(X_test)
    
    # Inverse transform the predictions to get the actual temperature values
    predictions = target_scaler.inverse_transform(predictions_scaled)

    # Calculate performance metrics
    rmse = math.sqrt(mean_squared_error(y_test_actual, predictions))
    mae = mean_absolute_error(y_test_actual, predictions)
    print(f'\n--- Evaluation on Separate Test Set ---')
    print(f'Test RMSE: {rmse:.2f}')
    print(f'Test MAE: {mae:.2f}')

    # --- 7. Visualize the Results ---
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(16, 8))

    ax.plot(df_test.index, y_test_actual, color='royalblue', label='Actual Temperature', linewidth=2)
    ax.plot(df_test.index, predictions, color='darkorange', linestyle='--', label='Predicted Temperature', linewidth=2)

    ax.set_title('Climate Forecast vs Actuals on Unseen Test Data', fontsize=20, pad=20)
    ax.set_xlabel('Date', fontsize=14)
    ax.set_ylabel('Mean Temperature (°C)', fontsize=14)
    ax.legend(loc='upper left', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"Error: A required data file was not found. Please ensure both '{TRAIN_FILE}' and '{TEST_FILE}' are in the correct directory.")
    print(f"Missing file: {e.filename}")
except Exception as e:
    print(f"An error occurred: {e}")



KeyboardInterrupt: 